# 03 - Creation et Entrainement du modele

Objectif : construire le CNN, le compiler et l'entrainer.

Fonctions testees : `conception_modele_cnn`, `compiler_modele`, `entrainer_modele`

In [2]:
import sys
from pathlib import Path
import pickle

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import chargement_pretraitement_dataset, preparer_donnees
from src.model_architecture import conception_modele_cnn, compiler_modele
from src.train import entrainer_modele

print("Imports OK")

2026-10-17 01:38:56.551626: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-17 01:38:56.587805: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-17 01:38:58.050354: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-17 01:39:04.714432: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Imports OK


In [3]:
# 1. Charger et preparer les donnees
DATA_PATH = PROJECT_ROOT / "data" / "raw"
X, y = chargement_pretraitement_dataset(str(DATA_PATH))
X_train, X_test, y_train, y_test, le, class_weights = preparer_donnees(X, y)

# Sauvegarder le LabelEncoder pour Streamlit
(PROJECT_ROOT / "models").mkdir(exist_ok=True)
with open(PROJECT_ROOT / "models" / "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print(f"LabelEncoder sauvegarde : {list(le.classes_)}")

Classes detectees : ['.gitkeep', 'glioma', 'meningioma', 'notumor', 'pituitary']
 -> Classe : glioma
 -> Classe : meningioma
 -> Classe : notumor
 -> Classe : pituitary

Chargement termine : 7023 images | {np.str_('notumor'), np.str_('pituitary'), np.str_('glioma'), np.str_('meningioma')}
Mapping : {np.str_('glioma'): np.int64(0), np.str_('meningioma'): np.int64(1), np.str_('notumor'): np.int64(2), np.str_('pituitary'): np.int64(3)}


: 

In [ ]:
# 2. Construire et compiler le modele
model = conception_modele_cnn(input_shape=(224, 224, 3), nb_classes=len(le.classes_))
model = compiler_modele(model, learning_rate=0.0001)
# Affiche le resume du modele (nombre de parametres)

In [ ]:
# 3. Entrainer
# epochs=5 pour un test rapide, mets 30 pour le vrai entrainement
history = entrainer_modele(model, X_train, y_train, X_test, y_test, class_weights, epochs=5, batch_size=32)
print("Entrainement termine !")
print(f"Modeles : models/best_model.keras et models/final_model.keras")

### Notes
- `best_model.keras` = meilleur modele sur val_accuracy
- `final_model.keras` = modele de la derniere epoch
- Utilise `best_model.keras` pour la prediction
- Change `epochs=30` pour un entrainement complet